# 📊 Notebook 01: Data Pipeline
## Fair Price Finder for Freelancers — Dataset V2

**Tim CC26-PSU164** — Coding Camp 2026 powered by DBS Foundation

**Tujuan Notebook:** Transformasi raw data dari 3 platform menjadi dataset siap training untuk AI Engineer.

**Input:**
- `data/raw/fastwork/fastwork_raw_v2.csv` — Fastwork raw data
- `data/raw/sribu/sribu_raw_v2.csv` — Sribu raw data
- `data/raw/projects/projects_raw_20260508_1320.csv` — Projects raw data

**Output:**
- `data/output/dataset_v2_features.csv` — Dataset siap training (numerik/encoded)
- `data/output/dataset_v2_text.csv` — Text mentah untuk eksperimen NLP (terpisah)
- `data/output/data_dictionary.csv` — Dokumentasi semua kolom

**Estimasi runtime:** ~2-3 menit

---
## ⚙️ Setup: Connect Google Drive

In [1]:
# Set path ke folder data root
# Notebook bisa dijalankan dari ai/notebooks atau dari root repo
from pathlib import Path
import os
import pandas as pd
import numpy as np
import re
from collections import Counter

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "data").exists():
    PROJECT_ROOT = NOTEBOOK_DIR / "data"
elif (NOTEBOOK_DIR.parent / "data").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent / "data"
elif (NOTEBOOK_DIR.parent.parent / "data").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent / "data"
else:
    PROJECT_ROOT = NOTEBOOK_DIR / "data"

# Pastikan struktur folder ada
 # Pastikan struktur folder ada
 # Definisikan OUTPUT_DIR lebih awal agar cell lain bisa mengaksesnya
OUTPUT_DIR = str(PROJECT_ROOT / "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cek isi folder data root
raw_files = sorted(PROJECT_ROOT.iterdir())
print(f'Data root: {PROJECT_ROOT}')
print('Files in data root:')
for f in raw_files:
    print(f'  - {f.name}')

Data root: C:\laragon\www\fair-price-finder\data
Files in data root:
  - cleaned
  - data_dictionary.md
  - final
  - output
  - raw
  - README.md


## 📚 Import Libraries

In [2]:
import pandas as pd
import numpy as np
import re
from collections import Counter

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

print('✅ Libraries loaded')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

✅ Libraries loaded
pandas version: 2.1.4
numpy version: 1.26.2


---
## 📥 STEP 1: Load Raw Data

In [3]:
RAW_PATHS = {
    'fastwork': PROJECT_ROOT / 'raw' / 'fastwork' / 'fastwork_raw_v2.csv',
    'sribu': PROJECT_ROOT / 'raw' / 'sribu' / 'sribu_raw_v2.csv',
    'projects': PROJECT_ROOT / 'raw' / 'projects' / 'projects_raw_20260508_1320.csv',
}

dfs = {}
for name, path in RAW_PATHS.items():
    df = pd.read_csv(path)
    df['source_platform'] = name
    dfs[name] = df
    print(f'{name:12s}: {len(df):,} rows × {df.shape[1]} cols')

total_raw = sum(len(d) for d in dfs.values())
print(f'\nTotal raw: {total_raw:,} rows')

fastwork    : 4,160 rows × 15 cols
sribu       : 1,181 rows × 15 cols
projects    : 123 rows × 15 cols

Total raw: 5,464 rows


In [4]:
# Preview Fastwork
print('=== FASTWORK SAMPLE ===')
dfs['fastwork'].head(3)

=== FASTWORK SAMPLE ===


,platform,kategori_utama,sub_kategori,judul_listing,harga,rating,jumlah_order,durasi_hari,skills,url_listing,harga_raw,rating_raw,jumlah_order_raw,durasi_raw,source_platform
0,fastwork,Web dan Pemrograman,Web Development,Pembuatan Website & Aplikasi Custom Profesiona...,3000000,4.9,237.0,6.0,React,https://fastwork.id/user/reynaldokwok/web-deve...,Rp3.000.000,"4,9 (133)",Terjual 237,Waktu pengerjaan 6 hari,fastwork
1,fastwork,Web dan Pemrograman,Web Development,"Website Company Profile, eCommerce, LMS, Sosme...",2500000,4.9,655.0,3.0,NaN,https://fastwork.id/user/maseyo/web-developmen...,Rp2.500.000,"4,9 (317)",Terjual 655,Waktu pengerjaan 3 hari,fastwork
2,fastwork,Web dan Pemrograman,Web Development,"Pembuatan Website Custom Professional ( React,...",1400000,4.9,60.0,5.0,"React, Laravel, WordPress",https://fastwork.id/user/imamrifai/web-develop...,Rp1.400.000,"4,9 (33)",Terjual 60,Waktu pengerjaan 5 hari,fastwork


In [5]:
# Preview Projects (punya deskripsi)
print('=== PROJECTS SAMPLE ===')
dfs['projects'].head(3)

=== PROJECTS SAMPLE ===


,platform,kategori_utama,sub_kategori,judul_listing,harga,durasi_hari,jumlah_bid,skills,url_listing,budget_raw,accepted_budget_raw,durasi_raw,deskripsi,tipe,source_platform
0,projects.co.id,Web dan Pemrograman,Website Development,Pembuatan Simple Website,650000,7,49.0,"Web Research, Web Programming, Website, Web De...",https://projects.co.id/public/browse_projects/...,"Rp 300,000 - 1,000,000",NaN,7 hari,Pekerjaan Pembuatan Simple Website (Scrip...,active,projects
1,projects.co.id,Web dan Pemrograman,Website Development,Bikin excel buat nota ada tanggal dan no urut,125000,2,17.0,Microsoft Excel,https://projects.co.id/public/browse_projects/...,"Rp 100,001 - 150,000",NaN,2 hari,halo semua freelancer saya ada project buat bi...,active,projects
2,projects.co.id,Web dan Pemrograman,Website Development,"[WP] Develop Web Komunitas, Events & Membership",1150000,7,9.0,"Wordpress, Website Building, Website Templates...",https://projects.co.id/public/browse_projects/...,"Rp 800,000 - 1,500,000",NaN,7 hari,Kami perlu dibantu slicing dari desain di figm...,active,projects


In [6]:
# Preview Sribu (punya deskripsi)
print('=== PROJECTS SAMPLE ===')
dfs['sribu'].head(3)

=== PROJECTS SAMPLE ===


,platform,kategori_utama,sub_kategori,judul_listing,harga,rating,jumlah_order,durasi_hari,skills,url_listing,harga_raw,rating_raw,jumlah_order_raw,durasi_raw,source_platform
0,sribu,Web dan Pemrograman,Web Development,Pembuatan Website Wordpress Murah dan Clean,99000.0,4.9,4.0,1.0,WordPress,https://www.sribu.com/id/users/emreadone/pembu...,Rp99.000,4.9,4Demo atau preview hasil,1 Hari Pengerjaan3 Revisi,sribu
1,sribu,Web dan Pemrograman,Web Development,"Pembuatan Website Custom - Laravel, Vue, React",500000.0,5.0,NaN,1.0,"React, Vue.js, Laravel",https://www.sribu.com/id/users/shafygunawan/pe...,Rp500.000,5.0,Ulasan dari Pembeli,Respon Chat8 jamProyek Berjalan1,sribu
2,sribu,Web dan Pemrograman,Web Development,Pembuatan Website Landing Page,99000.0,5.0,NaN,1.0,NaN,https://www.sribu.com/id/users/emreadone/pembu...,Rp99.000,5.0,Ulasan dari Pembeli,1 Hari Pengerjaan3 Revisi,sribu


---
## 🔧 STEP 2: Normalize Schema Per Platform

Setiap platform punya struktur kolom berbeda. Kita seragamkan ke schema standard:

In [7]:
def normalize_fastwork(df):
    """Normalize schema Fastwork."""
    out = pd.DataFrame()
    out['url_listing'] = df['url_listing']
    out['platform'] = 'fastwork'
    out['platform_type'] = 'service'
    out['kategori_utama'] = df['kategori_utama']
    out['sub_kategori'] = df['sub_kategori']
    out['judul_listing'] = df['judul_listing']
    out['deskripsi'] = ''  # Fastwork tidak punya deskripsi
    out['price_single'] = df['harga']
    out['price_min'] = df['harga']
    out['price_max'] = df['harga']
    out['durasi_hari'] = df['durasi_hari']
    out['rating'] = df['rating']
    out['has_rating'] = (df['rating'] > 0).astype(int)
    out['skills_raw'] = df['skills'].fillna('')
    return out

def normalize_sribu(df):
    """Normalize schema Sribu."""
    out = pd.DataFrame()
    out['url_listing'] = df['url_listing']
    out['platform'] = 'sribu'
    out['platform_type'] = 'service'
    out['kategori_utama'] = df['kategori_utama']
    out['sub_kategori'] = df['sub_kategori']
    out['judul_listing'] = df['judul_listing']
    out['deskripsi'] = ''
    out['price_single'] = df['harga']
    out['price_min'] = df['harga']
    out['price_max'] = df['harga']
    out['durasi_hari'] = df['durasi_hari']
    out['rating'] = df['rating']
    out['has_rating'] = (df['rating'] > 0).astype(int)
    out['skills_raw'] = df['skills'].fillna('')
    return out

def normalize_projects(df):
    """Normalize schema Projects.co.id (job-board, ada range budget + deskripsi)."""
    out = pd.DataFrame()
    out['url_listing'] = df['url_listing']
    out['platform'] = 'projects'
    out['platform_type'] = 'job_board'
    out['kategori_utama'] = df['kategori_utama']
    out['sub_kategori'] = df['sub_kategori']
    out['judul_listing'] = df['judul_listing']
    out['deskripsi'] = df['deskripsi'].fillna('')

    # Parse budget_raw 'Rp 300,000 - 1,000,000' → min/max
    def parse_budget(raw):
        if pd.isna(raw):
            return np.nan, np.nan
        nums = re.findall(r'[\d,\.]+', str(raw).replace('.', ''))
        nums = [int(n.replace(',', '')) for n in nums if n.replace(',', '').isdigit()]
        if len(nums) >= 2:
            return min(nums), max(nums)
        elif len(nums) == 1:
            return nums[0], nums[0]
        return np.nan, np.nan

    budget_parsed = df['budget_raw'].apply(parse_budget)
    out['price_min'] = [b[0] for b in budget_parsed]
    out['price_max'] = [b[1] for b in budget_parsed]
    out['price_single'] = (out['price_min'] + out['price_max']) / 2

    out['durasi_hari'] = df['durasi_hari']
    out['rating'] = np.nan
    out['has_rating'] = 0
    out['skills_raw'] = df['skills'].fillna('')
    return out

# Apply normalization
df_fw = normalize_fastwork(dfs['fastwork'])
df_sr = normalize_sribu(dfs['sribu'])
df_pr = normalize_projects(dfs['projects'])

df = pd.concat([df_fw, df_sr, df_pr], ignore_index=True)
print(f'Combined shape: {df.shape}')
df.head()

Combined shape: (5464, 14)


,url_listing,platform,platform_type,kategori_utama,sub_kategori,judul_listing,deskripsi,price_single,price_min,price_max,durasi_hari,rating,has_rating,skills_raw
0,https://fastwork.id/user/reynaldokwok/web-deve...,fastwork,service,Web dan Pemrograman,Web Development,Pembuatan Website & Aplikasi Custom Profesiona...,,3000000.0,3000000.0,3000000.0,6.0,4.9,1,React
1,https://fastwork.id/user/maseyo/web-developmen...,fastwork,service,Web dan Pemrograman,Web Development,"Website Company Profile, eCommerce, LMS, Sosme...",,2500000.0,2500000.0,2500000.0,3.0,4.9,1,
2,https://fastwork.id/user/imamrifai/web-develop...,fastwork,service,Web dan Pemrograman,Web Development,"Pembuatan Website Custom Professional ( React,...",,1400000.0,1400000.0,1400000.0,5.0,4.9,1,"React, Laravel, WordPress"
3,https://fastwork.id/user/arwebhouse/web-develo...,fastwork,service,Web dan Pemrograman,Web Development,Jasa Buat Sistem / Aplikasi Berbasis Web Untuk...,,1500000.0,1500000.0,1500000.0,2.0,4.8,1,
4,https://fastwork.id/user/adityawiryaa/web-deve...,fastwork,service,Web dan Pemrograman,Web Development,Jasa Pembuatan Website Profesional: Company Pr...,,2000000.0,2000000.0,2000000.0,7.0,5.0,1,


---
## 🗑️ STEP 3: Deduplication (True duplicates only)

Hapus duplikat berdasarkan URL listing (unique identifier asli per listing).

In [8]:
before = len(df)
df = df.drop_duplicates(subset=['url_listing'], keep='first').reset_index(drop=True)
after = len(df)
print(f'Before: {before:,}')
print(f'After:  {after:,}')
print(f'Removed: {before - after:,} true duplicates ({(before-after)/before*100:.1f}%)')

Before: 5,464
After:  5,104
Removed: 360 true duplicates (6.6%)


---
## 🛠️ STEP 4: Skill Extraction & Enrichment

Skill diambil dari 2 sumber:
1. Kolom `skills_raw` (kalau tersedia)
2. **Extract dari `judul_listing` + `deskripsi`** menggunakan keyword matching (untuk handle Fastwork yang 77% skill NaN)

In [9]:
# Skill keyword dictionary
SKILL_KEYWORDS = {
    # Web Development
    'wordpress': ['wordpress', 'wp'],
    'react': ['react', 'reactjs', 'react js', 'react.js'],
    'react native': ['react native'],
    'next.js': ['next.js', 'nextjs', 'next js'],
    'vue': ['vue.js', 'vuejs', 'vue js'],
    'laravel': ['laravel'],
    'flutter': ['flutter'],
    'kotlin': ['kotlin', 'android native'],
    'java': ['java spring', 'spring boot'],
    'php': ['php', 'codeigniter'],
    'javascript': ['javascript', ' js '],
    'python': ['python', 'django', 'flask'],
    'node.js': ['node.js', 'nodejs', 'node js'],
    'mysql': ['mysql', 'mariadb'],
    'html_css': ['html', 'css', 'tailwind', 'bootstrap'],

    # Design
    'figma': ['figma'],
    'photoshop': ['photoshop', 'adobe photoshop'],
    'illustrator': ['illustrator', 'adobe illustrator'],
    'canva': ['canva'],
    'adobe xd': ['adobe xd', 'xd'],
    'ui ux design': ['ui/ux', 'ui ux', 'uiux', 'user interface'],
    'logo design': ['logo design', 'desain logo'],
    'branding': ['brand identity', 'corporate identity', 'branding'],
    '3d modeling': ['3d', 'blender', 'maya', '3ds max'],

    # Marketing
    'seo': ['seo', 'search engine optimization'],
    'google ads': ['google ads', 'google adwords'],
    'facebook ads': ['facebook ads', 'fb ads'],
    'instagram': ['instagram marketing', 'ig ads', 'instagram ads'],
    'tiktok ads': ['tiktok', 'tiktok ads'],
    'meta ads': ['meta ads'],
    'copywriting': ['copywriting', 'copy writing'],
    'content writing': ['content writing', 'penulisan konten', 'artikel'],

    # Data & AI
    'machine learning': ['machine learning', 'ml '],
    'deep learning': ['deep learning'],
    'data science': ['data science'],
    'data analysis': ['data analysis', 'analisis data'],
    'computer vision': ['computer vision'],
    'nlp': ['nlp', 'natural language'],
    'tableau': ['tableau'],
    'power bi': ['power bi', 'powerbi'],
    'excel': ['microsoft excel', ' excel '],

    # Video & Audio
    'video editing': ['video editing', 'edit video'],
    'video production': ['video production', 'videography'],
    'after effects': ['after effects', 'motion graphics'],
    'voice over': ['voice over', 'voiceover'],
    'animation': ['animasi', 'animation'],

    # Writing
    'translation': ['translation', 'terjemah', 'penerjemahan'],
    'proofreading': ['proofreading', 'editing teks'],
}

print(f'Total skill keywords defined: {len(SKILL_KEYWORDS)}')

Total skill keywords defined: 48


In [10]:
SKILL_MIN_OCCURRENCES = 10  # Threshold include skill

def extract_skills_from_text(text):
    """Extract skill keywords dari title/deskripsi."""
    if pd.isna(text) or text == '':
        return set()
    text_lower = str(text).lower()
    found = set()
    for skill_name, keywords in SKILL_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            found.add(skill_name)
    return found

def parse_skills_raw(skills_str):
    """Parse skills_raw column into set."""
    if pd.isna(skills_str) or skills_str == '':
        return set()
    return set(s.strip().lower() for s in str(skills_str).split(',') if s.strip())

# Combine skills from raw column + extract from text
df['combined_text'] = df['judul_listing'].fillna('') + ' ' + df['deskripsi'].fillna('')

skills_per_row = []
for idx, row in df.iterrows():
    skills_from_raw = parse_skills_raw(row['skills_raw'])
    skills_from_text = extract_skills_from_text(row['combined_text'])
    skills_per_row.append(skills_from_raw | skills_from_text)

df['skills_combined'] = skills_per_row
df['skill_count'] = df['skills_combined'].apply(len)

# Count skill frequency
skill_counter = Counter()
for skills in skills_per_row:
    for s in skills:
        skill_counter[s] += 1

# Filter skill with min occurrences
valid_skills = [s for s, c in skill_counter.most_common() if c >= SKILL_MIN_OCCURRENCES]

print(f'Total unique skills: {len(skill_counter)}')
print(f'Skills meeting threshold (≥{SKILL_MIN_OCCURRENCES}): {len(valid_skills)}')
print(f'\nTop 20 skills:')
for skill in valid_skills[:20]:
    print(f'  {skill:25s}: {skill_counter[skill]:,}')

Total unique skills: 199
Skills meeting threshold (≥10): 45

Top 20 skills:
  seo                      : 311
  logo design              : 304
  video editing            : 223
  tiktok ads               : 213
  translation              : 193
  ui ux design             : 188
  content writing          : 157
  animation                : 123
  instagram                : 118
  wordpress                : 94
  machine learning         : 93
  data analysis            : 82
  figma                    : 79
  branding                 : 75
  excel                    : 73
  flutter                  : 68
  google ads               : 52
  python                   : 44
  canva                    : 43
  copywriting              : 42


In [11]:
# Create one-hot encoded columns untuk setiap valid skill
for skill in valid_skills:
    col_name = 'skill_' + skill.replace(' ', '_').replace('.', '').replace('/', '_')
    df[col_name] = df['skills_combined'].apply(lambda s: 1 if skill in s else 0)

# Premium skill flag
PREMIUM_SKILLS = {'machine learning', 'deep learning', 'data science', 'computer vision',
                  'nlp', 'flutter', 'react native', 'kotlin', 'next.js'}
df['has_premium_skill'] = df['skills_combined'].apply(
    lambda s: 1 if any(ps in s for ps in PREMIUM_SKILLS) else 0
)

print(f'✅ Created {len(valid_skills)} one-hot skill columns')
print(f'   Plus has_premium_skill flag')

✅ Created 45 one-hot skill columns
   Plus has_premium_skill flag


---
## 🏷️ STEP 5: Category Encoding

In [12]:
# Drop kategori noise
before = len(df)
df = df[df['kategori_utama'] != 'Lainnya'].reset_index(drop=True)
print(f'Dropped "Lainnya" category: {before - len(df)} rows')

# One-hot kategori_utama
kat_dummies = pd.get_dummies(df['kategori_utama'], prefix='kategori', dtype=int)
df = pd.concat([df, kat_dummies], axis=1)
print(f'Created {len(kat_dummies.columns)} kategori columns')

# One-hot platform
plat_dummies = pd.get_dummies(df['platform'], prefix='platform', dtype=int)
df = pd.concat([df, plat_dummies], axis=1)
print(f'Created {len(plat_dummies.columns)} platform columns')

# Platform type encoding
df['is_service_based'] = (df['platform_type'] == 'service').astype(int)

Dropped "Lainnya" category: 7 rows
Created 5 kategori columns
Created 3 platform columns


---
## ⭐ STEP 6: Handle Missing Rating

Projects.co.id (job-board) tidak punya rating. Strategi: impute dengan median rating per kategori.

In [13]:
rating_medians = df[df['has_rating']==1].groupby('kategori_utama')['rating'].median()
print('Rating median per kategori:')
for kat, med in rating_medians.items():
    print(f'  {kat:35s}: {med:.2f}')

global_median = df[df['has_rating']==1]['rating'].median()

def impute_rating(row):
    if row['has_rating'] == 1:
        return row['rating']
    return rating_medians.get(row['kategori_utama'], global_median)

df['rating_imputed'] = df.apply(impute_rating, axis=1)
print(f'\nImputed {(df["has_rating"]==0).sum():,} rows')

Rating median per kategori:
  Grafis & Desain                    : 5.00
  Pemasaran & Periklanan             : 4.90
  Penulisan & Penerjemahan           : 5.00
  Visual & Audio                     : 4.90
  Web dan Pemrograman                : 5.00

Imputed 2,145 rows


---
## ⚙️ STEP 7: Feature Engineering

In [14]:
# Log transform (target & durasi right-skewed)
df['log_price_single'] = np.log1p(df['price_single'])
df['log_price_min'] = np.log1p(df['price_min'])
df['log_price_max'] = np.log1p(df['price_max'])
df['log_durasi'] = np.log1p(df['durasi_hari'])

# Price range width (untuk Projects)
df['price_range_width'] = df['price_max'] - df['price_min']
df['has_price_range'] = (df['price_range_width'] > 0).astype(int)

# Text features
df['title_length'] = df['judul_listing'].fillna('').str.len()
df['title_word_count'] = df['judul_listing'].fillna('').str.split().str.len()
df['desc_length'] = df['deskripsi'].fillna('').str.len()
df['has_description'] = (df['desc_length'] > 0).astype(int)

# Urgency keyword detection
urgency_keywords = ['urgent', 'express', 'cepat', 'kilat', 'fast', 'asap']
df['has_urgency'] = df['judul_listing'].fillna('').str.lower().apply(
    lambda x: 1 if any(kw in x for kw in urgency_keywords) else 0
)

print('✅ Feature engineering complete:')
print('   - log_price_single, log_price_min, log_price_max, log_durasi')
print('   - price_range_width, has_price_range')
print('   - title_length, title_word_count, desc_length, has_description')
print('   - has_urgency')

✅ Feature engineering complete:
   - log_price_single, log_price_min, log_price_max, log_durasi
   - price_range_width, has_price_range
   - title_length, title_word_count, desc_length, has_description
   - has_urgency


---
## 🧹 STEP 8: Outlier & NaN Handling

In [15]:
before = len(df)

# 1. Drop NaN/0 price
mask_price = df['price_single'].notna() & (df['price_single'] > 0)
print(f'Drop NaN/0 price: {(~mask_price).sum()} rows')
df = df[mask_price]

# 2. Cap extreme prices (P1-P99 + hard cap 10K-100M)
p99 = df['price_single'].quantile(0.99)
p01 = df['price_single'].quantile(0.01)
df = df[(df['price_single'] >= max(p01, 10000)) & (df['price_single'] <= min(p99, 100_000_000))]
print(f'After price winsorization: {len(df):,} rows')

# 3. Impute durasi NaN dengan median per kategori
durasi_medians = df.groupby('kategori_utama')['durasi_hari'].median()
def impute_durasi(row):
    if pd.notna(row['durasi_hari']) and 0 < row['durasi_hari'] <= 365:
        return row['durasi_hari']
    return durasi_medians.get(row['kategori_utama'], df['durasi_hari'].median())
df['durasi_hari'] = df.apply(impute_durasi, axis=1)
df['log_durasi'] = np.log1p(df['durasi_hari'])  # Recompute
print('Imputed durasi NaN with median per kategori')

# 4. Drop listing dengan title tidak informatif
mask_title = df['judul_listing'].fillna('').str.len() >= 20
print(f'Drop listing dengan title <20 char: {(~mask_title).sum()} rows')
df = df[mask_title].reset_index(drop=True)

after = len(df)
print(f'\nBefore: {before:,} → After: {after:,} (-{before-after} rows, {(before-after)/before*100:.1f}% removed)')

Drop NaN/0 price: 10 rows
After price winsorization: 5,037 rows
Imputed durasi NaN with median per kategori
Drop listing dengan title <20 char: 396 rows

Before: 5,097 → After: 4,641 (-456 rows, 8.9% removed)


---
## 📦 STEP 9: Split into Features vs Text Files

In [16]:
skill_cols = [c for c in df.columns if c.startswith('skill_') and c != 'skill_count']
kategori_cols = [c for c in df.columns if c.startswith('kategori_') and c != 'kategori_utama']
platform_cols = [c for c in df.columns if c.startswith('platform_') and c != 'platform_type']

numeric_features = [
    # Target
    'price_single', 'price_min', 'price_max',
    'log_price_single', 'log_price_min', 'log_price_max',
    # Numeric inputs
    'durasi_hari', 'log_durasi',
    'rating_imputed', 'has_rating',
    'skill_count', 'has_premium_skill',
    'price_range_width', 'has_price_range',
    'title_length', 'title_word_count',
    'desc_length', 'has_description',
    'has_urgency', 'is_service_based',
] + kategori_cols + platform_cols + skill_cols

numeric_features = [c for c in numeric_features if c in df.columns]

# Dedup based on feature representation
before = len(df)
df = df.drop_duplicates(subset=numeric_features, keep='first').reset_index(drop=True)
print(f'Removed {before - len(df)} logical duplicates')

df['row_id'] = range(1, len(df) + 1)

# Build files
df_features = df[['row_id'] + numeric_features].copy()
df_text = df[['row_id', 'url_listing', 'platform', 'judul_listing', 'deskripsi', 'skills_raw']].copy()

print(f'\nFeatures file: {df_features.shape}')
print(f'Text file:     {df_text.shape}')

Removed 90 logical duplicates

Features file: (4551, 74)
Text file:     (4551, 6)


In [17]:
## 🔍 STEP 10b: Cek Missing Values & Duplicates — dataset_v2_features.csv

import pandas as pd

# Load dataset terbaru
df_check = pd.read_csv(f'{OUTPUT_DIR}/dataset_v2_features.csv')

print('='*60)
print('QUICK CHECK — dataset_v2_features.csv')
print('='*60)
print(f'Shape: {df_check.shape}')

# --- Duplicates ---
n_dup = df_check.drop(columns=['row_id']).duplicated().sum()
print(f'\n[DUPLICATES] {n_dup} rows duplikat {"✅ Aman" if n_dup == 0 else "⚠️ Ada duplikat!"}')

# --- Missing Values ---
missing = df_check.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print('[MISSING]    Tidak ada missing values ✅')
else:
    print(f'[MISSING]    {len(missing)} kolom punya NaN ⚠️:')
    for col, count in missing.items():
        pct = count / len(df_check) * 100
        print(f'  {col:35s}: {count:>5d} ({pct:.1f}%)')

QUICK CHECK — dataset_v2_features.csv
Shape: (4551, 74)

[DUPLICATES] 0 rows duplikat ✅ Aman
[MISSING]    Tidak ada missing values ✅


In [18]:
print('df index:', df.index.tolist()[:5])
print('df_text index:', df_text.index.tolist()[:5])
print('skills_combined sample:', df['skills_combined'].iloc[0])
print('skills_combined type:', type(df['skills_combined'].iloc[0]))

df index: [0, 1, 2, 3, 4]
df_text index: [0, 1, 2, 3, 4]
skills_combined sample: {'react', 'next.js'}
skills_combined type: <class 'set'>


In [19]:
## 🔍 STEP 10b: Cek Missing Values & Duplicates — dataset_v2_features.csv

import pandas as pd

# Load dataset terbaru
df_check = pd.read_csv(f'{OUTPUT_DIR}/dataset_v2_text.csv')

print('='*60)
print('QUICK CHECK — dataset_v2_text.csv')
print('='*60)
print(f'Shape: {df_check.shape}')

# --- Duplicates ---
n_dup = df_check.drop(columns=['row_id']).duplicated().sum()
print(f'\n[DUPLICATES] {n_dup} rows duplikat {"✅ Aman" if n_dup == 0 else "⚠️ Ada duplikat!"}')

# --- Missing Values ---
missing = df_check.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print('[MISSING]    Tidak ada missing values ✅')
else:
    print(f'[MISSING]    {len(missing)} kolom punya NaN ⚠️:')
    for col, count in missing.items():
        pct = count / len(df_check) * 100
        print(f'  {col:35s}: {count:>5d} ({pct:.1f}%)')

QUICK CHECK — dataset_v2_text.csv
Shape: (4551, 6)

[DUPLICATES] 0 rows duplikat ✅ Aman
[MISSING]    Tidak ada missing values ✅


In [20]:
# Cek berapa row yang skills_combined-nya empty set
print('Empty skills_combined:', (df['skills_combined'].apply(len) == 0).sum())

# Cek sample yang masih NaN di skills_raw
mask_nan = df_text['skills_raw'].isna() | (df_text['skills_raw'].str.strip() == '')
print('Masih kosong:', mask_nan.sum())
print(df_text[mask_nan][['judul_listing', 'skills_raw']].head(5))

Empty skills_combined: 2280
Masih kosong: 3528
                                       judul_listing skills_raw
1  Website Company Profile, eCommerce, LMS, Sosme...           
3  Jasa Buat Sistem / Aplikasi Berbasis Web Untuk...           
4  Jasa Pembuatan Website Profesional: Company Pr...           
5                   JASA PENGEMBANGAN APLIKASI & WEB           
6  Profesional Web Development & Website identity...           


---
## 💾 STEP 10: Save Output to Google Drive

In [21]:
## 💾 STEP 10: Save Output to Local
OUTPUT_DIR = f'{PROJECT_ROOT}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Isi NaN + empty string deskripsi → judul_listing
df_text['deskripsi'] = df_text['deskripsi'].apply(
    lambda x: x if pd.notna(x) and str(x).strip() != '' else None
)
df_text['deskripsi'] = df_text.apply(
    lambda row: row['judul_listing'] if pd.isna(row['deskripsi']) else row['deskripsi'],
    axis=1
)

# Isi NaN + empty string skills_raw → skills_combined, fallback ke judul_listing
skills_combined_str = df['skills_combined'].apply(lambda s: ', '.join(sorted(s)) if s else None)

df_text['skills_raw'] = df_text['skills_raw'].apply(
    lambda x: x if pd.notna(x) and str(x).strip() != '' else None
)
df_text['skills_raw'] = df_text['skills_raw'].fillna(skills_combined_str)

df_text['skills_raw'] = df_text.apply(
    lambda row: row['judul_listing'] if pd.isna(row['skills_raw']) else row['skills_raw'],
    axis=1
)

# Save
df_features.to_csv(f'{OUTPUT_DIR}/dataset_v2_features.csv', index=False)
df_text.to_csv(f'{OUTPUT_DIR}/dataset_v2_text.csv', index=False)

print('✅ Files saved locally:')
print(f'   → {os.path.abspath(OUTPUT_DIR)}/dataset_v2_features.csv ({df_features.shape})')
print(f'   → {os.path.abspath(OUTPUT_DIR)}/dataset_v2_text.csv ({df_text.shape})')

✅ Files saved locally:
   → C:\laragon\www\fair-price-finder\data\output/dataset_v2_features.csv ((4551, 74))
   → C:\laragon\www\fair-price-finder\data\output/dataset_v2_text.csv ((4551, 6))


---
## ✅ VALIDATION: Final Quality Check

In [22]:
print('='*60)
print('FINAL DATASET V2 — QUALITY SUMMARY')
print('='*60)
print(f'Shape: {df_features.shape}')
print(f'Duplicates: {df_features.drop(columns=["row_id"]).duplicated().sum()}')
print(f'NaN values: {df_features.isna().sum().sum()}')
print(f'Memory: {df_features.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

print('\n--- Platform Distribution ---')
for col in platform_cols:
    count = int(df_features[col].sum())
    print(f'  {col:30s}: {count:>5d} ({count/len(df_features)*100:.1f}%)')

print('\n--- Category Distribution ---')
for col in kategori_cols:
    count = int(df_features[col].sum())
    print(f'  {col:45s}: {count:>5d} ({count/len(df_features)*100:.1f}%)')

print('\n--- Price Statistics (IDR) ---')
print(df_features['price_single'].describe().round(0))

FINAL DATASET V2 — QUALITY SUMMARY
Shape: (4551, 74)
Duplicates: 0
NaN values: 0
Memory: 2.38 MB

--- Platform Distribution ---
  platform_fastwork             :  3386 (74.4%)
  platform_projects             :    86 (1.9%)
  platform_sribu                :  1079 (23.7%)

--- Category Distribution ---
  kategori_Grafis & Desain                     :   885 (19.4%)
  kategori_Pemasaran & Periklanan              :  1059 (23.3%)
  kategori_Penulisan & Penerjemahan            :   720 (15.8%)
  kategori_Visual & Audio                      :   546 (12.0%)
  kategori_Web dan Pemrograman                 :  1341 (29.5%)

--- Price Statistics (IDR) ---
count       4551.0
mean      579443.0
std       793730.0
min        50000.0
25%       150000.0
50%       300000.0
75%       650000.0
max      6500000.0
Name: price_single, dtype: float64


In [24]:
# Korelasi kunci fitur dengan target
key_features = ['log_durasi', 'has_premium_skill', 'skill_count', 'has_urgency',
                'is_service_based', 'has_rating', 'rating_imputed', 'title_length']

print('Korelasi dengan log_price_single:')
for f in key_features:
    corr = df_features[f].corr(df_features['log_price_single'])
    arrow = '↑' if corr > 0 else '↓'
    print(f'  {f:25s}: {corr:+.3f} {arrow}')

Korelasi dengan log_price_single:
  log_durasi               : +0.456 ↑
  has_premium_skill        : +0.189 ↑
  skill_count              : +0.064 ↑
  has_urgency              : -0.139 ↓
  is_service_based         : -0.095 ↓
  has_rating               : -0.046 ↓
  rating_imputed           : -0.012 ↓
  title_length             : +0.025 ↑


---
## 🎉 SELESAI!

Dataset v2 sudah tersimpan di Google Drive dan siap untuk:
1. **EDA** → lanjut ke notebook `02_eda_analysis.ipynb`
2. **Streamlit Dashboard** → lanjut ke notebook `03_streamlit_dashboard.ipynb`
3. **Model training** → handover ke AI Engineer

**File yang dihasilkan:**
- `output/dataset_v2_features.csv` — Dataset siap training
- `output/dataset_v2_text.csv` — Text mentah untuk NLP

**Next step:** Buka notebook `02_eda_analysis.ipynb` untuk lanjut ke fase Exploratory Data Analysis.